# Task 2 — Advanced EDA and Text Mining

**Dataset status:** No raw Task 2 dataset was attached to the assignment text in this chat. This notebook is a reusable analysis pipeline; it will inspect the dataset you place at `DATA_PATH`. Do not present outputs from the synthetic demonstration as findings about HanuAI's real data.

In [ ]:
# If required, install packages:
%pip install pandas numpy matplotlib seaborn scikit-learn openpyxl

In [ ]:
from pathlib import Path
import re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.cluster import KMeans
from sklearn.decomposition import NMF

DATA_PATH = Path("provided_dataset.xlsx")  # change to the supplied CSV/XLSX file
OUTPUT_DIR = Path("task2_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def load_dataset(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset not found at {path.resolve()}. Put the provided dataset here or update DATA_PATH."
        )
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(path)
    raise ValueError("Supported formats are CSV and Excel (.xlsx/.xls).")

df = load_dataset(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

## 1. Data quality and column profile

This section summarizes types, missingness, duplicates, unique values, and basic numeric distributions. Review the report before deciding whether to drop or impute values: missing data may itself be operationally meaningful.

In [ ]:
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_non_null": df.nunique(dropna=True),
})
display(profile.sort_values("missing_pct", ascending=False))
print("Duplicate full rows:", int(df.duplicated().sum()))
display(df.describe(include="all").T)
profile.to_csv(OUTPUT_DIR / "column_profile.csv", index=True)

# Conservative cleanup: preserve original df; use a separate clean copy.
clean = df.drop_duplicates().copy()
# Normalize whitespace in text columns without deleting source columns.
text_cols = clean.select_dtypes(include=["object", "string"]).columns.tolist()
for col in text_cols:
    clean[col] = clean[col].map(lambda x: re.sub(r"\\s+", " ", x).strip() if isinstance(x, str) else x)
print("Rows after exact duplicate removal:", len(clean))

## 2. Distributions and critical columns

Use plots to inspect the columns that matter to the domain. Automatically generated plots are exploratory; confirm units and definitions with the data owner.

In [ ]:
numeric_cols = clean.select_dtypes(include=np.number).columns.tolist()
for col in numeric_cols[:8]:
    ax = clean[col].dropna().plot(kind="hist", bins=30, title=f"Distribution: {col}")
    ax.set_xlabel(col)
    plt.tight_layout()
    plt.show()

# Frequency tables for categorical columns with manageable cardinality.
cat_cols = [c for c in text_cols if clean[c].nunique(dropna=True) <= 30]
for col in cat_cols[:8]:
    display(clean[col].fillna("[Missing]").value_counts().head(20).rename_axis(col).to_frame("count"))

## 3. Identify free-text columns and extract tags

The heuristic below identifies likely narrative fields by average text length. Edit `TEXT_COLUMNS` after reviewing the actual schema. Rule-based tags are transparent and easy to validate; they are not a substitute for domain review.

In [ ]:
text_candidates = []
for col in text_cols:
    values = clean[col].dropna().astype(str)
    avg_len = values.str.len().mean() if len(values) else 0
    if avg_len >= 25 or values.str.split().str.len().mean() >= 5:
        text_candidates.append(col)
print("Likely free-text columns:", text_candidates)

# Set this explicitly after inspecting the dataset. Example: ["description", "remarks", "issue_text"]
TEXT_COLUMNS = text_candidates[:]
if not TEXT_COLUMNS:
    print("No likely free-text fields found. Inspect columns and set TEXT_COLUMNS manually.")

# Example tag dictionary: extend with vocabulary observed in the real dataset.
TAG_RULES = {
    "component_failure": [r"\\b(fail(?:ure|ed)?|broken|damaged|crack(?:ed)?|malfunction)\\b"],
    "electrical_issue": [r"\\b(electric(?:al)?|short circuit|voltage|current|wiring|power surge)\\b"],
    "connectivity_issue": [r"\\b(network|connect(?:ion|ivity)|offline|signal|communication|not responding)\\b"],
    "battery_or_power": [r"\\b(battery|charging|power supply|low power|discharg(?:e|ed))\\b"],
    "overheating": [r"\\b(overheat(?:ing)?|too hot|temperature high|thermal)\\b"],
    "installation_or_setup": [r"\\b(install(?:ation)?|setup|commission(?:ing)?|configuration|mounting)\\b"],
    "maintenance_or_wear": [r"\\b(maintenance|wear|corrosion|rust|loose|replacement)\\b"],
    "safety_risk": [r"\\b(safety|hazard|unsafe|shock|fire|smoke|injury)\\b"],
}
compiled_rules = {tag: [re.compile(p, re.I) for p in patterns] for tag, patterns in TAG_RULES.items()}

def extract_tags(value):
    if not isinstance(value, str):
        return []
    return [tag for tag, patterns in compiled_rules.items() if any(p.search(value) for p in patterns)]

if TEXT_COLUMNS:
    clean["combined_text"] = clean[TEXT_COLUMNS].fillna("").astype(str).agg(" ".join, axis=1)
    clean["extracted_tags"] = clean["combined_text"].apply(extract_tags)
    clean["issue_type"] = clean["extracted_tags"].apply(lambda tags: tags[0] if tags else "unclassified")
    clean["extracted_tags_string"] = clean["extracted_tags"].apply(lambda tags: "; ".join(tags))
    display(clean[TEXT_COLUMNS + ["extracted_tags_string", "issue_type"]].head())
    clean.to_csv(OUTPUT_DIR / "text_mining_tags.csv", index=False, encoding="utf-8-sig")
else:
    print("Skipping tag export until text columns are configured.")

## 4. Topic discovery / clustering

TF-IDF + KMeans gives a simple baseline for grouping similar descriptions. Choose the number of clusters after reviewing sample text and stability. Clusters are not proven root causes; validate with domain experts. This section requires at least a few non-empty text records.

In [ ]:
def cluster_texts(text_series, n_clusters=5):
    texts = text_series.fillna("").astype(str)
    usable = texts.str.strip().ne("")
    if usable.sum() < max(10, n_clusters):
        raise ValueError(f"Need at least {max(10, n_clusters)} non-empty text rows for this baseline.")
    vectorizer = TfidfVectorizer(stop_words="english", max_features=3000, ngram_range=(1, 2), min_df=2)
    X = vectorizer.fit_transform(texts[usable])
    model = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    output = pd.DataFrame({"row_index": texts[usable].index, "cluster": labels, "text": texts[usable].values})
    terms = np.array(vectorizer.get_feature_names_out())
    top_terms = {}
    for cluster_id, center in enumerate(model.cluster_centers_):
        top_terms[int(cluster_id)] = terms[center.argsort()[-10:][::-1]].tolist()
    return output, top_terms

if "combined_text" in clean.columns:
    try:
        clustered, cluster_terms = cluster_texts(clean["combined_text"], n_clusters=min(5, max(2, len(clean)//10)))
        clustered.to_csv(OUTPUT_DIR / "text_clusters.csv", index=False, encoding="utf-8-sig")
        print(json.dumps(cluster_terms, indent=2))
    except ValueError as exc:
        print("Clustering skipped:", exc)

## 5. Stakeholder insight checklist

After running on the real dataset, answer these using measured results and examples:
1. Which issue tags are most frequent, and how do counts change over time?
2. Which assets, locations, product versions, or components are associated with the highest rate/severity?
3. Are missing fields or duplicate records concentrated in specific sources or periods?
4. What representative records support each cluster/topic?
5. Which actions are low-cost and high-impact, and what KPI would confirm improvement?

Do not infer causation from frequency alone. Compare against exposure/usage denominators where possible, account for reporting bias, and validate tags/clusters with subject-matter experts.

In [ ]:
# Export cleaned data and tag summary when applicable.
clean.to_csv(OUTPUT_DIR / "cleaned_dataset.csv", index=False, encoding="utf-8-sig")
if "extracted_tags" in clean.columns:
    tag_counts = clean["extracted_tags"].explode().dropna().value_counts().rename_axis("tag").reset_index(name="count")
    tag_counts.to_csv(OUTPUT_DIR / "tag_summary.csv", index=False)
    display(tag_counts)
print(f"Outputs written to: {OUTPUT_DIR.resolve()}")